# Agent Transaction Data Cleaning

## Project Overview

This notebook explores, cleans, and validates agent transaction
data to improve its quality and reliability.

The dataset contains agent information and transaction records.
The cleaning process addresses duplicates, inconsistent transaction
types, missing values, suspicious amounts, date formatting, and
unknown agent references.

In [1]:
import pandas as pd
import numpy as np

# Load the original datasets
agents = pd.read_csv("agents.csv")

txns = pd.read_csv(
    "agent_transactions.csv",
    dtype={"amount": "string"}
)

print("Agents dataset:", agents.shape)
print("Transactions dataset:", txns.shape)

Agents dataset: (20, 3)
Transactions dataset: (612, 5)


In [2]:
# First five agent records
display(agents.head())

# First five transaction records
display(txns.head())

,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [3]:
print("AGENTS INFORMATION")
agents.info()

print("\nTRANSACTIONS INFORMATION")
txns.info()

AGENTS INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   agent_id   20 non-null     str  
 1   county     20 non-null     str  
 2   onboarded  20 non-null     str  
dtypes: str(3)
memory usage: 612.0 bytes

TRANSACTIONS INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   txn_id    612 non-null    str   
 1   agent_id  612 non-null    str   
 2   txn_type  612 non-null    str   
 3   amount    590 non-null    string
 4   txn_date  612 non-null    str   
dtypes: str(4), string(1)
memory usage: 24.0 KB


In [4]:
print("Missing values in agents:")
display(agents.isnull().sum())

print("Missing values in transactions:")
display(txns.isnull().sum())

Missing values in agents:


agent_id     0
county       0
onboarded    0
dtype: int64

Missing values in transactions:


txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64

In [5]:
print("Duplicate transaction IDs:")
print(txns["txn_id"].duplicated().sum())

print("Duplicate transaction records:")
print(txns.duplicated().sum())

Duplicate transaction IDs:
12
Duplicate transaction records:
11


In [6]:
print("Transaction types and their frequencies:")
display(txns["txn_type"].value_counts(dropna=False))

Transaction types and their frequencies:


txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

In [7]:
print("Amount values:")
display(txns["amount"].value_counts(dropna=False).head(15))

print("N/A amounts:", txns["amount"].str.strip().str.upper().eq("N/A").sum())

print("250000 amounts:", txns["amount"].str.strip().eq("250000").sum())

Amount values:


amount
<NA>      22
250000     5
2501.0     3
2383.0     3
1455.0     3
893.0      3
2583.0     2
2798.0     2
1755.0     2
2330.0     2
1462.0     2
301.0      2
2143.0     2
1007.0     2
1062.0     2
Name: count, dtype: Int64

N/A amounts: 0
250000 amounts: 5


In [8]:
print("Dates containing forward slashes:")
print(txns["txn_date"].str.contains("/", na=False).sum())

display(txns[txns["txn_date"].str.contains("/", na=False)].head())

Dates containing forward slashes:
5


,txn_id,agent_id,txn_type,amount,txn_date
87,TX00088,AG014,deposit,7072.0,2026/04/27
128,TX00129,AG010,deposit,1620.0,2026/04/28
181,TX00182,AG003,DEPOSIT,533.0,2026/03/20
437,TX00438,AG008,DEPOSIT,1518.0,2026/03/30
571,TX00572,AG011,Withdrawal,1750.0,2026/03/04


In [9]:
unknown_agents = txns[
    ~txns["agent_id"].isin(agents["agent_id"])
]

print("Transactions linked to unknown agents:", len(unknown_agents))
display(unknown_agents.head())

Transactions linked to unknown agents: 8


,txn_id,agent_id,txn_type,amount,txn_date
55,TX00056,AG099,DEPOSIT,8507.0,2026-04-01
68,TX00069,AG099,withdrawal,1877.0,2026-04-24
77,TX00078,AG099,deposit,1770.0,2026-04-06
237,TX00238,AG099,DEPOSIT,4944.0,2026-04-06
267,TX00268,AG099,DEPOSIT,3110.0,2026-04-09


In [10]:
# Display all records with duplicated transaction IDs

duplicate_ids = txns[
    txns["txn_id"].duplicated(keep=False)
].sort_values("txn_id")

display(duplicate_ids)

,txn_id,agent_id,txn_type,amount,txn_date
23,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
605,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
66,TX00067,AG002,deposit,<NA>,2026-04-07
608,TX00067,AG002,deposit,<NA>,2026-04-07
165,TX00166,AG010,deposit,4146.0,2026-04-18
609,TX00166,AG010,deposit,4146.0,2026-04-18
201,TX00202,AG010,Deposit,2243.0,2026-03-22
604,TX00202,AG010,Deposit,2243.0,2026-03-22
611,TX00330,AG016,withdrawal,1219.0,2026-04-07
329,TX00330,AG016,withdrawal,1219.0,2026-04-07


In [11]:
# Find transaction IDs associated with different records

conflicting_ids = (
    txns.groupby("txn_id")
    .filter(lambda group: len(group) > 1)
)

display(conflicting_ids)

,txn_id,agent_id,txn_type,amount,txn_date
23,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
66,TX00067,AG002,deposit,<NA>,2026-04-07
165,TX00166,AG010,deposit,4146.0,2026-04-18
201,TX00202,AG010,Deposit,2243.0,2026-03-22
329,TX00330,AG016,withdrawal,1219.0,2026-04-07
334,TX00335,AG014,Deposit,<NA>,2026-03-11
404,TX00405,AG014,Deposit,1936.0,2026-04-03
415,TX00416,AG015,deposit,4664.0,2026-03-19
446,TX00447,AG001,Withdrawal,1607.0,2026-03-01
455,TX00456,AG003,withdrawal,1284.0,2026-03-14


In [12]:
# Count exact duplicate rows
exact_duplicates = txns.duplicated().sum()

# Count duplicated transaction IDs
duplicate_id_count = txns["txn_id"].duplicated().sum()

# Find IDs associated with different records
conflicting_duplicate_ids = (
    txns.groupby("txn_id")
    .filter(lambda group: len(group) > 1)
)

print("Exact duplicate rows:", exact_duplicates)
print("Duplicate transaction IDs:", duplicate_id_count)

print(
    "Unique duplicated transaction IDs:",
    conflicting_duplicate_ids["txn_id"].nunique()
)

Exact duplicate rows: 11
Duplicate transaction IDs: 12
Unique duplicated transaction IDs: 12


In [13]:
# Find duplicated IDs with more than one distinct agent ID

agent_conflicts = (
    txns.groupby("txn_id")["agent_id"]
    .nunique()
)

agent_conflicts = agent_conflicts[agent_conflicts > 1]

print("Transaction IDs with conflicting agents:")
display(txns[txns["txn_id"].isin(agent_conflicts.index)])

Transaction IDs with conflicting agents:


,txn_id,agent_id,txn_type,amount,txn_date
404,TX00405,AG014,Deposit,1936.0,2026-04-03
601,TX00405,AG099,Deposit,1936.0,2026-04-03


## Stage 4: Agent Data Cleaning and Quality Assessment

### 1. Introduction
This stage involved cleaning and validating agent transaction records to improve data quality, consistency, and reliability. Python and Pandas were used to develop a reusable data-cleaning pipeline.

### 2. Data Cleaning Activities
The following operations were performed:

- **Duplicate removal:** Removed 11 exact duplicate transaction records while retaining conflicting transaction IDs for further investigation.
- **Transaction type standardization:** Converted transaction types to lowercase and removed unnecessary whitespace.
- **Amount validation:** Converted transaction amounts into numeric values and identified missing and suspicious amounts.
- **Date standardization:** Converted transaction dates into a consistent datetime format.
- **Agent ID validation:** Identified transactions referencing agents who do not exist in the agent master dataset.
- **Transaction validation:** Checked transaction IDs, agent IDs, transaction types, dates, and amounts for missing or invalid values.

### 3. Data Quality Assessment

The original transaction dataset contained 612 records. After removing exact duplicates, 601 records remained.

| Quality issue | Records identified |
|---|---:|
| Exact duplicates removed | 11 |
| Missing transaction IDs | 0 |
| Missing agent IDs | 0 |
| Missing transaction types | 0 |
| Missing transaction dates | 0 |
| Missing or invalid amounts | 20 |
| Unknown agent references | 8 |
| Suspicious amounts (KES 250,000) | 5 |

### 4. Key Findings
- Eight transactions reference an unknown agent ID, AG099.
- Twenty transactions have missing amounts.
- Five transactions contain suspicious amounts of KES 250,000 and require further investigation.
- Transaction ID TX00405 has conflicting agent references. Both records were retained to avoid losing potentially important information.
- Transaction types and dates were successfully standardized.

### 5. Output Files
The following files were generated and saved in the `outputs` directory:

1. `agent_transactions_clean.csv` – contains the cleaned and validated transaction records, including quality flags.
2. `agent_data_quality_report.csv` – contains a summary of the identified data quality issues.

### 6. Conclusion
The cleaning pipeline successfully improved the consistency and reliability of the agent transaction dataset. Remaining data quality issues were flagged for further investigation rather than being silently deleted or modified. The reusable pipeline and quality report provide a foundation for subsequent data analysis.

In [14]:
import importlib
import cleaning_pipeline

importlib.reload(cleaning_pipeline)

from cleaning_pipeline import clean_agent_data

clean_df, cleaning_log = clean_agent_data(txns, agents)

print("Cleaned records:", len(clean_df))
print("Duplicate transaction IDs:", clean_df["txn_id"].duplicated().sum())
print("Unknown agents:", clean_df["unknown_agent"].sum())
print("Imputed amounts:", clean_df["amount_imputed"].sum())
print("Outliers:", clean_df["is_outlier"].sum())

display(cleaning_log)

Cleaned records: 600
Duplicate transaction IDs: 0
Unknown agents: 7
Imputed amounts: 20
Outliers: 53


,action,rows_affected,details
0,Remove exact duplicates,11,Removed identical transaction records.
1,Resolve duplicate transaction IDs,1,"Retained one record per transaction ID, prefer..."
2,Standardize transaction types,0,Standardized valid values to Deposit and Withd...
3,Identify missing or invalid amounts,20,Converted invalid strings and missing values t...
4,Impute missing amounts,20,Replaced missing amounts with median: 1776.50.
5,Flag outliers,53,IQR bounds: -1540.25 to 5513.75.
6,Validate transaction dates,0,Converted transaction dates to datetime.
7,Identify unknown agents,7,Flagged transactions referencing unregistered ...
8,Merge agent register,7,Left join used to retain transactions with unk...


In [15]:
# Inspect the suspicious KES 250,000 transactions
print("KES 250,000 transactions:")

display(
    clean_df.loc[
        clean_df["amount"] == 250000,
        ["txn_id", "agent_id", "txn_type", "amount", "is_outlier"]
    ]
)

# Count outliers by amount
print("\nOutlier summary:")

display(
    clean_df.loc[clean_df["is_outlier"]]
    .groupby("amount")
    .size()
    .sort_values(ascending=False)
    .head(15)
)

# Check the IQR bounds
print("\nIQR statistics:")

print("Q1:", clean_df["amount"].quantile(0.25))
print("Q3:", clean_df["amount"].quantile(0.75))
print("Upper bound:", clean_df["amount"].quantile(0.75) +
      1.5 * (
          clean_df["amount"].quantile(0.75) -
          clean_df["amount"].quantile(0.25)
      ))

KES 250,000 transactions:


,txn_id,agent_id,txn_type,amount,is_outlier
68,TX00425,AG005,Withdrawal,250000.0,True
100,TX00307,AG012,Deposit,250000.0,True
109,TX00316,AG014,Deposit,250000.0,True
199,TX00595,AG016,Deposit,250000.0,True
256,TX00464,AG009,Withdrawal,250000.0,True



Outlier summary:


amount
250000.0    5
5567.0      1
5615.0      1
5632.0      1
5643.0      1
5646.0      1
5745.0      1
5749.0      1
5777.0      1
5788.0      1
5856.0      1
5897.0      1
6006.0      1
6158.0      1
6235.0      1
dtype: int64


IQR statistics:
Q1: 1105.0
Q3: 2868.5
Upper bound: 5513.75


## Data Quality Audit

In [16]:
# 1. Duplicate transaction IDs
duplicate_rows = txns["txn_id"].duplicated(keep=False).sum()

# 2. Inconsistent transaction types
standardized_types = (
    txns["txn_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "deposit": "Deposit",
        "withdrawal": "Withdrawal"
    })
)

inconsistent_types = (
    txns["txn_type"].astype("string") != standardized_types
).sum()

# 3. Missing or invalid amounts
numeric_amounts = pd.to_numeric(
    txns["amount"],
    errors="coerce"
)

missing_amounts = numeric_amounts.isna().sum()

# 4. Suspicious KES 250,000 amounts
suspicious_amounts = numeric_amounts.eq(250000).sum()

# 5. Inconsistent date formats
inconsistent_dates = (
    txns["txn_date"].astype("string").str.contains("/", na=False)
).sum()

# 6. Unknown agent references
unknown_agents = (
    ~txns["agent_id"].isin(agents["agent_id"])
).sum()

# 7. IQR outliers identified by the cleaning pipeline
iqr_outliers = clean_df["is_outlier"].sum()

# Build the audit table
audit_data = [
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": duplicate_rows
    },
    {
        "issue": "Inconsistent transaction types",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": inconsistent_types
    },
    {
        "issue": "Inconsistent date formats",
        "dimension": "Consistency",
        "column": "txn_date",
        "rows_affected": inconsistent_dates
    },
    {
        "issue": "Missing or invalid amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": missing_amounts
    },
    {
        "issue": "IQR outliers",
        "dimension": "Validity",
        "column": "amount",
        "rows_affected": iqr_outliers
    },
    {
        "issue": "Suspicious KES 250,000 amounts",
        "dimension": "Accuracy",
        "column": "amount",
        "rows_affected": suspicious_amounts
    },
    {
        "issue": "Unknown agent references",
        "dimension": "Referential Integrity",
        "column": "agent_id",
        "rows_affected": unknown_agents
    }
]

audit_table = pd.DataFrame(audit_data)

display(audit_table)

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,24
1,Inconsistent transaction types,Consistency,txn_type,384
2,Inconsistent date formats,Consistency,txn_date,5
3,Missing or invalid amounts,Completeness,amount,22
4,IQR outliers,Validity,amount,53
5,"Suspicious KES 250,000 amounts",Accuracy,amount,5
6,Unknown agent references,Referential Integrity,agent_id,8


## Agent Merge and Validation

In [17]:
# Validate the cleaned and merged dataset

print("Original transaction rows:", len(txns))
print("Cleaned transaction rows:", len(clean_df))

print(
    "Duplicate transaction IDs:",
    clean_df["txn_id"].duplicated().sum()
)

print(
    "Transactions with unknown agents:",
    clean_df["unknown_agent"].sum()
)

print(
    "Transactions without a county:",
    clean_df["county"].isna().sum()
)

print("\nCounty distribution:")

display(
    clean_df["county"].value_counts(dropna=False)
)

Original transaction rows: 612
Cleaned transaction rows: 600
Duplicate transaction IDs: 0
Transactions with unknown agents: 7
Transactions without a county: 7

County distribution:


county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
NaN          7
Name: count, dtype: int64

In [18]:
# Convert ISO date strings back to datetime for analysis
clean_df["txn_date"] = pd.to_datetime(clean_df["txn_date"])

# County-level summary by transaction type
county_summary = (
    clean_df
    .groupby(["county", "txn_type"], dropna=False)
    .agg(
        total_value=("amount", "sum"),
        number_of_transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
)

print("COUNTY SUMMARY")
display(county_summary)

# Create week column
clean_df["week"] = (
    clean_df["txn_date"]
    .dt.to_period("W")
    .apply(lambda period: str(period.start_time.date()))
)

# Weekly transaction value by county
weekly_pivot = pd.pivot_table(
    clean_df,
    index="week",
    columns="county",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

print("WEEKLY TRANSACTION VALUE BY COUNTY")
display(weekly_pivot)

COUNTY SUMMARY


,county,txn_type,total_value,number_of_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.25
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.5
9,Nakuru,Withdrawal,328594.5,39,8425.5


WEEKLY TRANSACTION VALUE BY COUNTY


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru
week,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2743.5
2026-03-02,18317.0,287933.0,285669.5,68967.0,56574.0
2026-03-09,267771.5,23186.0,30477.5,52860.5,28730.0
2026-03-16,22054.0,28125.0,25144.5,43107.5,39795.0
2026-03-23,11504.0,27077.0,17050.0,41373.5,18998.5
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0
2026-04-06,15030.0,15341.0,47092.5,56786.5,276609.0
2026-04-13,12265.0,29416.0,26730.0,56432.5,26152.5
2026-04-20,24507.0,37439.0,16687.0,65187.0,30862.0


In [19]:
# Create a separate summary dataset
summary_df = clean_df.copy()

# Keep transactions with unknown counties visible
summary_df["county"] = summary_df["county"].fillna("Unknown")

# County-level summary
county_summary = (
    summary_df
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        number_of_transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
)

print("COUNTY SUMMARY")
display(county_summary)

# Weekly transaction value by county
weekly_pivot = pd.pivot_table(
    summary_df,
    index="week",
    columns="county",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

print("WEEKLY TRANSACTION VALUE BY COUNTY")
display(weekly_pivot)

COUNTY SUMMARY


,county,txn_type,total_value,number_of_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.25
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.5
9,Nakuru,Withdrawal,328594.5,39,8425.5


WEEKLY TRANSACTION VALUE BY COUNTY


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


### Business Interpretation

The county-level analysis shows that Nairobi recorded the highest deposit transaction value at KES 543,590, while Mombasa recorded the highest withdrawal transaction value at KES 368,592. Nakuru recorded the highest average withdrawal value at KES 8,425.50, suggesting that its withdrawal transactions tend to be larger on average. Kiambu and Kisumu recorded higher deposit values than withdrawal values, while Mombasa and Nakuru showed higher withdrawal values than deposits.

The weekly pivot table provides a comparison of transaction values across counties over time. Transactions linked to unknown agents are retained under an Unknown category to avoid excluding records from the analysis. These findings can help management identify regional transaction patterns and areas requiring further investigation.

In [20]:
import os

# Create outputs directory
os.makedirs("outputs", exist_ok=True)

# Ensure transaction dates use ISO format
clean_df["txn_date"] = pd.to_datetime(
    clean_df["txn_date"]
).dt.strftime("%Y-%m-%d")

# Export the final cleaned dataset
clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Final agent transactions exported successfully!")

Final agent transactions exported successfully!


In [21]:
print("Rows:", len(clean_df))

print(
    "Duplicate transaction IDs:",
    clean_df["txn_id"].duplicated().sum()
)

print(
    "Transaction types:",
    sorted(clean_df["txn_type"].dropna().unique())
)

print(
    "Amount data type:",
    clean_df["amount"].dtype
)

print(
    "Date format example:",
    clean_df["txn_date"].iloc[0]
)

print(
    "Imputed amount column:",
    "amount_imputed" in clean_df.columns
)

print(
    "Outlier flag column:",
    "is_outlier" in clean_df.columns
)

print(
    "County column:",
    "county" in clean_df.columns
)

Rows: 600
Duplicate transaction IDs: 0
Transaction types: ['Deposit', 'Withdrawal']
Amount data type: Float64
Date format example: 2026-03-03
Imputed amount column: True
Outlier flag column: True
County column: True


# Data Quality Report — Mobile Money Agent Network

## 1. Overview

The original transaction dataset contained 612 records linked to 20 registered agents across five counties. Data profiling identified duplicate transaction IDs, inconsistent transaction types, missing amounts, suspicious values, inconsistent date formats and references to unknown agents. A reusable Python cleaning pipeline was developed to address these issues. After cleaning and merging with the agent register, 600 transaction records remained, with no duplicate transaction IDs.

## 2. Data Quality Assessment

### Completeness
The original dataset contained 22 missing or invalid transaction amounts. After duplicate removal, 20 records still had missing amounts. These were imputed using the median transaction amount of KES 1,776.50. An `amount_imputed` flag was added to identify the affected records.

### Uniqueness
The original dataset contained 12 duplicated transaction ID groups involving 24 rows. Eleven exact duplicate rows were removed, and one conflicting duplicate transaction ID was resolved. For TX00405, the record linked to registered agent AG014 was retained instead of the record linked to unknown agent AG099. The final dataset contains 600 unique transaction IDs.

### Consistency
Transaction types appeared in different capitalization and whitespace formats. A standardization process converted them into exactly two categories: Deposit and Withdrawal. Five transaction dates using forward slashes were also converted into the ISO date format YYYY-MM-DD.

### Validity
The IQR method identified 53 unusually high transaction amounts using an upper bound of KES 5,513.75 and a lower bound of -KES 1,540.25. An `is_outlier` flag was added. Outliers were retained rather than automatically deleted to avoid losing potentially valid transactions.

### Accuracy
Five transactions contained the suspicious amount of KES 250,000. These values are considerably higher than the typical transaction amounts and may indicate data-entry errors or unusual transactions. They were flagged for investigation and retained pending verification against the original transaction records.

### Referential Integrity
Eight original transaction references pointed to unknown agent AG099. One was resolved when the conflicting TX00405 record was handled, leaving seven unknown-agent transactions. A left join preserved these transactions, while their county values remained missing and were represented as Unknown in summary tables.

## 3. Business Findings

Nairobi recorded the highest deposit value at KES 543,590, while Mombasa recorded the highest withdrawal value at KES 368,592. Nakuru recorded the highest average withdrawal value at KES 8,425.50. Weekly transaction summaries provide additional insight into the distribution of transaction values across counties.

## 4. Privacy and Data Protection

If the extract also contained customer phone numbers, those numbers would constitute personal data because they could identify individual customers. Transaction records linked to those numbers could also reveal information about individuals' financial activities. Under Kenya's Data Protection Act, 2019, the organisation should process such information lawfully and transparently, collect only what is necessary, and protect it against unauthorized access. Phone numbers should be masked or pseudonymised for analytical work, encrypted when stored or transmitted, and accessible only to authorized personnel. The organisation should also apply appropriate retention periods, access controls and secure deletion procedures.

## 5. Conclusion

The cleaning process improved the reliability, consistency and usability of the agent transaction dataset. The final dataset contains 600 records, unique transaction IDs, standardized transaction types, numeric amounts, ISO-formatted dates, quality flags and agent county information. Unresolved anomalies were retained and documented for further investigation rather than silently deleted or altered. The cleaning log provides an auditable record of the operations performed.